# 02 — Diffusion analysis

This notebook estimates diffusion coefficients from preprocessed TrackMate trajectories.

Input generated by `01_trackmate_preprocessing.ipynb`:

```text
results/preprocessed/filtered_spots_combined.csv
```

The notebook performs:
- mean squared displacement (MSD) calculation,
- linear fitting of the initial MSD region,
- diffusion coefficient estimation using `D = slope / 4`,
- per-track and per-cell summary generation,
- basic diffusion QC plots.

Velocity distributions, Rayleigh reference curves, HMM, and clustering are handled in downstream notebooks.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.style.use("default")


CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RESULTS_DIR = PROJECT_DIR / "results"
PREPROCESSED_DIR = RESULTS_DIR / "preprocessed"
DIFFUSION_DIR = RESULTS_DIR / "diffusion"

FIGURES_DIR = PROJECT_DIR / "figures"
DIFFUSION_FIGURES_DIR = FIGURES_DIR / "diffusion"

for directory in [
    RESULTS_DIR,
    PREPROCESSED_DIR,
    DIFFUSION_DIR,
    FIGURES_DIR,
    DIFFUSION_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


FILTERED_SPOTS_PATH = PREPROCESSED_DIR / "filtered_spots_combined.csv"

CONDITION_ORDER = [
    "control",
    "500uM_H2O2",
    "500uM_H2O2_30min",
]

CONDITION_LABELS = {
    "control": "Control",
    "500uM_H2O2": "500 µM H₂O₂",
    "500uM_H2O2_30min": "500 µM H₂O₂ 30 min",
}

N_FIT_POINTS = 3
MIN_MSD_POINTS = 3

print("Project directory:", PROJECT_DIR)
print("Input file:", FILTERED_SPOTS_PATH)
print("Configuration loaded")


## 2. Load preprocessed trajectories

The input table must contain filtered localizations after edge-artifact removal and track-length filtering.


In [ ]:
if not FILTERED_SPOTS_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed file was not found: {FILTERED_SPOTS_PATH}. "
        "Run 01_trackmate_preprocessing.ipynb first."
    )

spots = pd.read_csv(FILTERED_SPOTS_PATH)

required_columns = [
    "protein",
    "condition",
    "sample",
    "cell",
    "file_name",
    "track_id",
    "x",
    "y",
    "t",
    "frame",
]

missing_columns = [
    column for column in required_columns
    if column not in spots.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(spots.columns)}"
    )

spots = spots.sort_values(
    ["protein", "sample", "cell", "condition", "track_id", "frame"]
).reset_index(drop=True)

print("Loaded spots:", len(spots))
print("Number of tracks:", spots["track_id"].nunique())

spots.head()


## 3. MSD and diffusion functions

In [ ]:
def compute_msd_for_track(track):
    """Compute time-averaged MSD for a single trajectory."""
    track = track.sort_values("frame")

    x = track["x"].to_numpy(dtype=float)
    y = track["y"].to_numpy(dtype=float)
    t = track["t"].to_numpy(dtype=float)
    frame = track["frame"].to_numpy(dtype=int)

    n_points = len(track)

    if n_points < 2:
        return pd.DataFrame()

    rows = []

    for lag in range(1, n_points):
        dx = x[lag:] - x[:-lag]
        dy = y[lag:] - y[:-lag]

        squared_displacement = dx**2 + dy**2
        dt = t[lag:] - t[:-lag]
        tau_s = np.nanmedian(dt)

        if not np.isfinite(tau_s) or tau_s <= 0:
            frame_dt = frame[lag:] - frame[:-lag]
            tau_s = np.nanmedian(frame_dt)

        rows.append({
            "lag_frames": lag,
            "tau_s": tau_s,
            "msd_um2": np.nanmean(squared_displacement),
            "n_displacements": len(squared_displacement),
        })

    return pd.DataFrame(rows)


def estimate_diffusion_coefficient(msd, n_fit_points=3):
    """Estimate D from initial MSD points.

    For two-dimensional Brownian diffusion:

        MSD(t) = 4Dt

    Therefore:

        D = slope / 4
    """
    if msd.empty or len(msd) < MIN_MSD_POINTS:
        return np.nan, np.nan, np.nan, np.nan

    fit_data = msd.dropna(subset=["tau_s", "msd_um2"]).copy()
    fit_data = fit_data[
        (fit_data["tau_s"] > 0)
        & (fit_data["msd_um2"] >= 0)
    ]

    if len(fit_data) < MIN_MSD_POINTS:
        return np.nan, np.nan, np.nan, np.nan

    fit_data = fit_data.head(n_fit_points)

    if len(fit_data) < MIN_MSD_POINTS:
        return np.nan, np.nan, np.nan, np.nan

    tau = fit_data["tau_s"].to_numpy(dtype=float)
    msd_values = fit_data["msd_um2"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(tau, msd_values, deg=1)
    diffusion_coefficient = slope / 4

    predicted = slope * tau + intercept
    residual_sum_of_squares = np.sum((msd_values - predicted) ** 2)
    total_sum_of_squares = np.sum((msd_values - np.mean(msd_values)) ** 2)

    if total_sum_of_squares > 0:
        r_squared = 1 - residual_sum_of_squares / total_sum_of_squares
    else:
        r_squared = np.nan

    return diffusion_coefficient, slope, intercept, r_squared


## 4. Estimate diffusion coefficients for all trajectories

In [ ]:
track_results = []
msd_results = []

group_columns = [
    "protein",
    "condition",
    "sample",
    "cell",
    "file_name",
    "track_id",
]

for keys, track in spots.groupby(group_columns, sort=False):
    protein, condition, sample, cell, file_name, track_id = keys

    msd = compute_msd_for_track(track)

    if msd.empty:
        continue

    diffusion_coefficient, slope, intercept, r_squared = estimate_diffusion_coefficient(
        msd,
        n_fit_points=N_FIT_POINTS,
    )

    track_length = len(track)
    total_displacement = np.sqrt(
        (track["x"].iloc[-1] - track["x"].iloc[0]) ** 2
        + (track["y"].iloc[-1] - track["y"].iloc[0]) ** 2
    )

    track_results.append({
        "protein": protein,
        "condition": condition,
        "sample": sample,
        "cell": cell,
        "file_name": file_name,
        "track_id": track_id,
        "track_length": track_length,
        "total_displacement_um": total_displacement,
        "D_um2_s": diffusion_coefficient,
        "log10_D": np.log10(diffusion_coefficient)
        if np.isfinite(diffusion_coefficient) and diffusion_coefficient > 0
        else np.nan,
        "msd_slope": slope,
        "msd_intercept": intercept,
        "msd_fit_r2": r_squared,
        "n_fit_points": N_FIT_POINTS,
    })

    msd = msd.copy()
    msd["protein"] = protein
    msd["condition"] = condition
    msd["sample"] = sample
    msd["cell"] = cell
    msd["file_name"] = file_name
    msd["track_id"] = track_id

    msd_results.append(msd)

track_results = pd.DataFrame(track_results)
msd_results = pd.concat(msd_results, ignore_index=True) if msd_results else pd.DataFrame()

valid_track_results = track_results[
    np.isfinite(track_results["D_um2_s"])
    & (track_results["D_um2_s"] > 0)
].copy()

track_results_path = DIFFUSION_DIR / "per_track_diffusion.csv"
valid_track_results_path = DIFFUSION_DIR / "per_track_diffusion_valid.csv"
msd_results_path = DIFFUSION_DIR / "per_track_msd.csv"

track_results.to_csv(track_results_path, index=False)
valid_track_results.to_csv(valid_track_results_path, index=False)
msd_results.to_csv(msd_results_path, index=False)

print("All tracks:", len(track_results))
print("Valid D tracks:", len(valid_track_results))
print("Saved:", track_results_path)
print("Saved:", valid_track_results_path)
print("Saved:", msd_results_path)

valid_track_results.head()


## 5. Per-cell summary statistics

In [ ]:
cell_summary = (
    valid_track_results.groupby(
        ["protein", "condition", "sample", "cell", "file_name"],
        observed=True,
    )
    .agg(
        n_tracks=("track_id", "count"),
        median_D_um2_s=("D_um2_s", "median"),
        mean_D_um2_s=("D_um2_s", "mean"),
        median_log10_D=("log10_D", "median"),
        mean_log10_D=("log10_D", "mean"),
        median_track_length=("track_length", "median"),
        median_total_displacement_um=("total_displacement_um", "median"),
    )
    .reset_index()
)

cell_summary_path = DIFFUSION_DIR / "cell_level_diffusion_summary.csv"
cell_summary.to_csv(cell_summary_path, index=False)

print("Saved:", cell_summary_path)

cell_summary.sort_values(
    ["protein", "sample", "cell", "condition"],
    na_position="last",
)


## 6. Plotting functions

In [ ]:
def setup_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)


def get_available_conditions(data):
    return [
        condition for condition in CONDITION_ORDER
        if condition in data["condition"].unique()
    ]


def plot_diffusion_boxplot(protein):
    data = valid_track_results[
        valid_track_results["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No valid diffusion data for {protein}")
        return

    conditions = get_available_conditions(data)
    values = [
        data.loc[data["condition"] == condition, "D_um2_s"].dropna().to_numpy()
        for condition in conditions
    ]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.boxplot(
        values,
        labels=[CONDITION_LABELS[c] for c in conditions],
        showfliers=False,
    )

    ax.set_yscale("log")
    ax.set_ylabel("D, µm²/s")
    ax.set_title(f"{protein}: diffusion coefficient distribution")
    setup_axis(ax)
    plt.xticks(rotation=20)
    plt.tight_layout()

    output_path = DIFFUSION_FIGURES_DIR / f"{protein}_diffusion_boxplot.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_cell_level_diffusion(protein):
    data = cell_summary[cell_summary["protein"] == protein].copy()

    if data.empty:
        print(f"No cell-level diffusion data for {protein}")
        return

    data["condition"] = pd.Categorical(
        data["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    data = data.sort_values(["sample", "cell", "condition"])

    x_map = {condition: i for i, condition in enumerate(CONDITION_ORDER)}

    fig, ax = plt.subplots(figsize=(7, 5))

    for (sample, cell), group in data.groupby(["sample", "cell"], observed=True):
        group = group.sort_values("condition")
        x = group["condition"].map(x_map).astype(float).to_numpy()
        y = group["median_D_um2_s"].to_numpy()

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            alpha=0.55,
            label=f"S{sample} cell{cell}",
        )

    median_summary = (
        data.groupby("condition", observed=True)["median_D_um2_s"]
        .median()
        .reindex(CONDITION_ORDER)
    )

    ax.plot(
        range(len(CONDITION_ORDER)),
        median_summary.values,
        marker="o",
        linewidth=4,
        color="black",
        label="group median",
    )

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("Median D per cell, µm²/s")
    ax.set_title(f"{protein}: cell-level diffusion summary")
    setup_axis(ax)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
    plt.tight_layout()

    output_path = DIFFUSION_FIGURES_DIR / f"{protein}_cell_level_diffusion.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_cumulative_diffusion(protein):
    data = valid_track_results[
        valid_track_results["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No valid diffusion data for {protein}")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    for condition in get_available_conditions(data):
        values = data.loc[
            data["condition"] == condition,
            "D_um2_s",
        ].dropna().to_numpy()

        values = values[values > 0]

        if len(values) == 0:
            continue

        values = np.sort(values)
        cumulative = np.arange(1, len(values) + 1) / len(values)

        ax.plot(
            values,
            cumulative,
            linewidth=2,
            label=CONDITION_LABELS[condition],
        )

    ax.set_xscale("log")
    ax.set_xlabel("D, µm²/s")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title(f"{protein}: cumulative diffusion distribution")
    setup_axis(ax)
    ax.legend(frameon=False)
    plt.tight_layout()

    output_path = DIFFUSION_FIGURES_DIR / f"{protein}_cumulative_diffusion.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


## 7. Generate diffusion figures

In [ ]:
proteins = sorted(valid_track_results["protein"].dropna().unique())

for protein in proteins:
    plot_diffusion_boxplot(protein)
    plot_cell_level_diffusion(protein)
    plot_cumulative_diffusion(protein)


## 8. Quick interpretation guide

- `per_track_diffusion_valid.csv` contains diffusion estimates for individual trajectories with physically meaningful positive D values.
- `cell_level_diffusion_summary.csv` is the recommended table for condition-level comparison because the cell, not the individual trajectory, is the biological/statistical unit.
- Track-level distributions are useful for visualization of heterogeneity but should not be used alone for statistical inference because trajectories within the same cell are not independent.


## 9. Output files

This notebook generates:

```text
results/diffusion/per_track_diffusion.csv
results/diffusion/per_track_diffusion_valid.csv
results/diffusion/per_track_msd.csv
results/diffusion/cell_level_diffusion_summary.csv
figures/diffusion/*_diffusion_boxplot.png
figures/diffusion/*_cell_level_diffusion.png
figures/diffusion/*_cumulative_diffusion.png
```

These outputs are used by downstream notebooks for velocity analysis, mobility-state classification, sensitivity analysis, and final figure generation.
